# 04 超参调优与实验管理

> 前置：`03-training-engineering`、`04-neural-networks/06-optimizers-in-practice`。
> 目标：把"凭感觉调参"变成"系统搜索 + 交叉验证 + 实验留痕"。调参不是玄学，是**用有限算力做最有效的搜索**。

## 三层调参

| 层次 | 方法 | 成本 | 适用 |
|------|------|------|------|
| L1 | 手写网格/随机搜索 | 低 | 2~3 个超参 |
| L2 | sklearn GridSearchCV / RandomizedSearchCV | 中 | 常规模型 |
| L3 | 贝叶斯优化（Optuna 等） | 中高 | 昂贵训练、连续超参 |

**核心规则：一切超参选择必须用交叉验证（或验证集），绝不能用测试集调参**——否则测试集就"脏"了，最终分数虚高。

In [ ]:
# 本模块通用导入（全部 CPU 即可运行）
import os
import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
import sklearn
import warnings
warnings.filterwarnings("ignore")

# 中文字体兼容（Windows / macOS）
plt.rcParams["font.sans-serif"] = ["Microsoft YaHei", "SimHei", "PingFang SC", "DejaVu Sans"]
plt.rcParams["axes.unicode_minus"] = False

print("numpy", np.__version__, "| pandas", pd.__version__, "| sklearn", sklearn.__version__)

In [ ]:
from sklearn.datasets import load_digits
from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV, RandomizedSearchCV, learning_curve
from sklearn.ensemble import RandomForestClassifier

# MNIST 的轻量版（8x8 手写数字）——Kaggle Digit Recognizer 的离线替身
X, y = load_digits(return_X_y=True)
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.2, random_state=0, stratify=y)
print("digits：", X.shape, "  类别数 =", len(np.unique(y)), "  训练", X_tr.shape[0], "测试", X_te.shape[0])

# L1 手写网格搜索：2 个超参 × 4 组合，每组 5 折交叉验证
grid = [(n, d) for n in [50, 200] for d in [None, 8]]
print("\n手写网格搜索（每组 5 折 CV）：")
for n, d in grid:
    m = RandomForestClassifier(n_estimators=n, max_depth=d, random_state=0)
    s = cross_val_score(m, X_tr, y_tr, cv=5, scoring="accuracy")
    print(f"n_estimators={n:4d} max_depth={str(d):4s}  CV准确率 = {s.mean():.4f} ± {s.std():.4f}")
best_manual = max(((n, d) for n, d in grid),
                  key=lambda nd: cross_val_score(RandomForestClassifier(n_estimators=nd[0], max_depth=nd[1], random_state=0),
                                                 X_tr, y_tr, cv=5, scoring="accuracy").mean())
print("手写网格最优：", best_manual)

## 网格 vs 随机搜索

**网格搜索**穷举所有组合——超参多时组合爆炸（4 个超参各 10 档 = 10⁴ 次训练）。

**随机搜索**（Bergstra & Bengio, 2012）按分布随机采样组合：同样的预算下，**随机搜索通常更优**，因为每个超参的重要性不均匀，随机采样能更全面覆盖"重要超参"的取值空间。

**交叉验证为什么必要**：单次 train/val 划分的分数方差大（运气成分）；K 折 CV 把数据切 K 份、轮流做验证、分数取平均，估计更稳：

$$\text{CV}_k = \frac{1}{K}\sum_{k=1}^{K} \text{score}_k$$

In [ ]:
# L2 sklearn 网格搜索
gs = GridSearchCV(RandomForestClassifier(random_state=0),
                  {"n_estimators": [50, 200], "max_depth": [None, 8]},
                  cv=5, scoring="accuracy")
gs.fit(X_tr, y_tr)
print("GridSearchCV 最优：", gs.best_params_, "  CV得分 =", round(gs.best_score_, 4))

# L2 随机搜索：更大空间、固定预算
rs = RandomizedSearchCV(RandomForestClassifier(random_state=0),
                        {"n_estimators": [50, 100, 200],
                         "max_depth": [None, 4, 8, 16],
                         "min_samples_split": [2, 5, 10]},
                        n_iter=8, cv=5, random_state=0, scoring="accuracy")
rs.fit(X_tr, y_tr)
print("RandomizedSearchCV 最优：", rs.best_params_, "  CV得分 =", round(rs.best_score_, 4))
print("最优模型在测试集得分（仅最后看一眼）：", round(rs.score(X_te, y_te), 4))

## 学习曲线：判断欠拟合 / 过拟合

- **欠拟合**：训练集与 CV 两条线都低、且接近 → 模型太弱，加容量或加特征；
- **过拟合**：训练集远高于 CV → 加正则、减容量、加数据；
- **理想**：两线都高、差距小。

"数据量 × 模型容量"的匹配关系，决定你该调模型还是该采数据。

In [ ]:
sizes, train_s, val_s = learning_curve(
    RandomForestClassifier(n_estimators=100, random_state=0), X_tr, y_tr,
    train_sizes=[0.2, 0.4, 0.6, 0.8, 1.0], cv=5, scoring="accuracy")
plt.figure(figsize=(6.5, 4))
plt.plot(sizes, train_s.mean(1), "o-", label="训练集准确率")
plt.plot(sizes, val_s.mean(1), "s--", label="交叉验证准确率")
plt.xlabel("训练样本数"); plt.ylabel("准确率")
plt.title("学习曲线：判断欠拟合（两线低且近）vs 过拟合（两线差距大）")
plt.legend(); plt.grid(alpha=0.3); plt.tight_layout(); plt.show()

# 判断示例：这里是"容量足够、数据越多越好"的健康形态
gap = (train_s.mean(1) - val_s.mean(1))[-1]
print(f"最终训练/CV 差距 = {gap:.4f} → 差距小说明不过拟合，继续加数据仍会涨")

## 实验管理：留痕 = 可复现 = 可信

工业/竞赛的标准做法：
1. **一条记录 = 一次实验**：配置（超参、数据版本、代码版本）+ 指标（CV 分数、测试分数）；
2. **seed 矩阵**：同一配置跑 3~5 个 seed 取均值方差——单次结果可能是运气；
3. **工具**：小项目用 CSV/字典（本课演示），大项目用 Weights & Biases（wandb）/ MLflow，它们就是"实验留痕 + 图表对比"的自动化版。

原则：**任何一次实验，三个月后别人（或你自己）必须能按记录完全复现。**

In [ ]:
# 实验记录：配置 + 指标写入 DataFrame 并导出（CSV 就是最小可用的实验管理系统）
records = []
for n, d in grid:
    m = RandomForestClassifier(n_estimators=n, max_depth=d, random_state=0)
    s = cross_val_score(m, X_tr, y_tr, cv=5, scoring="accuracy")
    records.append({"model": "RandomForest", "n_estimators": n, "max_depth": d,
                    "cv_mean": round(s.mean(), 4), "cv_std": round(s.std(), 4)})
log = pd.DataFrame(records).sort_values("cv_mean", ascending=False)
print(log.to_string(index=False))
log.to_csv("experiment_log.csv", index=False)
print("\n已保存 experiment_log.csv —— 实验管理底线：每个配置都有记录、可复现")

# seed 矩阵：同一配置 3 个 seed，看稳定性
seeds = []
for sd in [0, 1, 2]:
    m = RandomForestClassifier(n_estimators=200, max_depth=8, random_state=sd)
    seeds.append(cross_val_score(m, X_tr, y_tr, cv=5, scoring="accuracy").mean())
print("\n同配置 3 个 seed 的 CV 分数：", [round(s, 4) for s in seeds],
      "  均值 =", round(np.mean(seeds), 4), "  波动 =", round(np.std(seeds), 4))

## 贝叶斯优化（概念）

L3 方法用**代理模型**（如高斯过程）拟合"超参 → 分数"的关系，选择**采集函数**（如期望改进 EI）最大的点去试，而非盲目随机：

$$EI(x) = \mathbb{E}[\max(0,\ f(x) - f_{\text{best}})]$$

每试一次，代理模型更新一次，越试越聪明。Optuna 就是开箱即用的实现（`pip install optuna`）。表格模型场景下，**随机搜索 + 合理预算通常已足够**，贝叶斯优化在高昂训练（DL/LLM）时才值得上。

## 课后练习（Kaggle）

1. **Digit Recognizer 调参**（<https://www.kaggle.com/c/digit-recognizer>）：把本课搜索方法用在 MNIST（7 万样本）上，给随机森林或逻辑回归做调参，提交 **≥ 0.97**（提示：加 PCA 降维后调参）。
2. **House Prices 特征+调参**（<https://www.kaggle.com/c/house-prices-advanced-regression-techniques>）：调 `RandomForestRegressor` 的 `n_estimators / max_depth / min_samples_split`，用 RMSE 交叉验证选优，记录完整实验日志（CSV），目标提交 RMSE < 0.16。
3. **思考题**：为什么调参必须用交叉验证而不能用测试集分数？如果测试集被你调过 20 次，最终分数意味着什么？（答案：它已不再是"未知数据"的估计，你的 0.99 是过拟合测试集的结果。）